<a href="https://colab.research.google.com/github/yc-115/programing-language/blob/main/%E3%80%8CHW1_%E6%97%A5%E5%B8%B8%E6%94%AF%E5%87%BA%E9%80%9F%E7%AE%97%E8%88%87%E5%88%86%E6%94%A4_Gradio_ipynb%E3%80%8D41371211H.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#日常支出速算與分攤（作業一）
- 目標：從 Sheet 讀「消費紀錄」→ 計總額/分類小計/AA 分攤 → 寫回 Sheet Summary 分頁。
- AI 點子（可選）：請模型總結本週花錢習慣與建議（例如「外食過多」）。
- Sheet 欄位：date, category, item, amount, payer

GoogleSheet: https://docs.google.com/spreadsheets/d/1amoN-IbD2JvGg-uTmj1sK37abhYgkaapv2PJNP8JCfs/edit?usp=sharing

In [8]:
import gradio as gr
import pandas as pd
import datetime
import pytz  # 用於處理時區
import gspread
from google.colab import auth
from google.auth import default

In [9]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1amoN-IbD2JvGg-uTmj1sK37abhYgkaapv2PJNP8JCfs/edit?usp=sharing"
WORKSHEET_NAME = "工作表1"
REQUIRED_COLUMNS = ["日期", "時間", "分類", "品項", "金額", "付款人", "地點", "支付方式", "備註"]

tw_tz = pytz.timezone('Asia/Taipei')

_auth_done = False
_gc = None
_gs = None

In [12]:
def _ensure_auth():
    global _auth_done, _gc, _gs
    if not _auth_done:
        auth.authenticate_user()
        creds, _ = default()
        _gc = gspread.authorize(creds)
        _gs = _gc.open_by_url(SHEET_URL)
        _auth_done = True
    return _gs

def _get_ws(name):
    gs = _ensure_auth()
    try:
        ws = gs.worksheet(name)
        existing_headers = ws.row_values(1)
        if existing_headers != REQUIRED_COLUMNS:
            ws.update(values=[REQUIRED_COLUMNS], range_name='A1')
        return ws
    except:
        ws = gs.add_worksheet(title=name, rows="1000", cols="20")
        ws.update(values=[REQUIRED_COLUMNS], range_name='A1')
        return ws

def get_now_tw():
    now = datetime.datetime.now(tw_tz)
    return now.strftime('%Y-%m-%d'), now.strftime('%H:%M')

def add_expense_fast(date, time, cat, item, amt, payer, loc, method, note):
    try:
        ws = _get_ws(WORKSHEET_NAME)

        # 格式化日期
        try:
            date_obj = pd.to_datetime(date.replace('/', '-'))
            formatted_date = date_obj.strftime('%Y-%m-%d')
        except:
            formatted_date = date

        try:
            amt_val = float(amt)
        except:
            return "⚠️ 金額必須是數字", None, None, None

        # 依照順序組裝
        new_row = [
            formatted_date, time, cat or "未填",
            item or "未填", amt_val, payer or "匿名",
            loc or "", method or "", note or ""
        ]

        ws.append_row(new_row, value_input_option='USER_ENTERED')

        # 成功後自動更新 UI 的日期時間為「下一刻」的台灣時間
        new_date, new_time = get_now_tw()
        return f"✅ 已成功新增：{item} (${amt_val})", new_date, new_time, gr.update(), gr.update()
    except Exception as e:
        return f"❌ 錯誤: {str(e)}", gr.update(), gr.update(), None, None

def refresh_and_deduplicate():
    try:
        ws = _get_ws(WORKSHEET_NAME)
        all_values = ws.get_all_values()

        if not all_values or len(all_values) <= 1:
            return "📭 目前尚無資料", 0, None, None, pd.DataFrame(columns=REQUIRED_COLUMNS)

        df = pd.DataFrame(all_values[1:], columns=all_values[0])
        for col in REQUIRED_COLUMNS:
            if col not in df.columns:
                df[col] = ""

        df = df[REQUIRED_COLUMNS]
        df["金額"] = pd.to_numeric(df["金額"].astype(str).str.replace(',', ''), errors="coerce").fillna(0.0)

        df_unique = df.drop_duplicates()
        if len(df) != len(df_unique):
            ws.clear()
            ws.update(values=[REQUIRED_COLUMNS] + df_unique.fillna('').values.tolist(), range_name='A1')
            msg = "✅ 已同步並自動清除重複資料"
        else:
            msg = "✅ 資料已同步"

        total = float(df_unique["金額"].sum())
        cat_df = df_unique.groupby("分類", as_index=False)["金額"].sum().sort_values("金額", ascending=False)

        # AA 結算邏輯
        payers = [p for p in df_unique["付款人"].unique() if p and p != "匿名"]
        if payers:
            share = total / len(payers)
            paid = df_unique.groupby("付款人", as_index=False)["金額"].sum().rename(columns={"金額": "實付"})
            paid["應付(AA)"] = share
            paid["差額"] = paid["實付"] - paid["應付(AA)"]
            settle_df = paid
        else:
            settle_df = pd.DataFrame(columns=["付款人", "差額"])

        return msg, total, cat_df, settle_df, df_unique
    except Exception as e:
        return f"❌ 同步失敗: {str(e)}", 0, None, None, None

with gr.Blocks(title="雲端記帳台灣時間版") as demo:
    init_date, init_time = get_now_tw()

    gr.Markdown(f"## 🧾 日常支出速算與分攤（Gradio）\n- 新增支出後自動寫回 Google Sheet\n- 一鍵查看總額、分類小計與 AA 分攤\n- 讀寫工作表：`工作表1`")

    with gr.Tab("➕ 新增支出"):
        with gr.Row():
            date_in = gr.Textbox(label="日期 (YYYY-MM-DD)", value=init_date)
            time_in = gr.Textbox(label="時間 (HH:MM)", value=init_time)
        with gr.Row():
            cat_in = gr.Textbox(label="分類")
            item_in = gr.Textbox(label="品項")
            amt_in = gr.Textbox(label="金額")
        with gr.Row():
            payer_in = gr.Textbox(label="付款人")
            loc_in = gr.Textbox(label="地點")
            method_in = gr.Textbox(label="支付方式")
        note_in = gr.Textbox(label="備註")

        add_btn = gr.Button("立即儲存", variant="primary")
        msg_out = gr.Markdown()

    with gr.Tab("📊 統計與同步"):
        sync_btn = gr.Button("🔄 同步資料並校正")
        with gr.Row():
            total_num = gr.Number(label="總支出金額")
            cat_view = gr.Dataframe(label="分類摘要")
        settle_view = gr.Dataframe(label="AA 結算")
        raw_view = gr.Dataframe(label="原始資料預覽")

    # 按鈕動作
    add_btn.click(
        fn=add_expense_fast,
        inputs=[date_in, time_in, cat_in, item_in, amt_in, payer_in, loc_in, method_in, note_in],
        outputs=[msg_out, date_in, time_in, total_num, cat_view]
    )

    sync_btn.click(
        fn=refresh_and_deduplicate,
        outputs=[msg_out, total_num, cat_view, settle_view, raw_view]
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0cda5dbb48e9a03a6d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
